In [1]:
# ==============================================================================
# CELL 1: Environment Setup & Dependencies
# RUN THIS FIRST
# ==============================================================================

import os
import sys

# MUST be set before any torch imports
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess

print("📦 Installing dependencies (this may take 5-10 minutes)...")
print("   Note: Pinning speechbrain<1.0 to avoid lazy import bug\n")

# Clean up any broken installs
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                       "speechbrain", "whisperx", "pyannote.audio", "faster-whisper"])

# Install with compatible versions
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch",
    "torchaudio",
    "faster-whisper",
    "whisperx",
    "pyannote.audio",
    "speechbrain<1.0",
    "yt-dlp",
    "pandas",
    "imageio-ffmpeg",
    "requests",
])

print("✅ Dependencies installed\n")


📦 Installing dependencies (this may take 5-10 minutes)...
   Note: Pinning speechbrain<1.0 to avoid lazy import bug

✅ Dependencies installed



In [ ]:
# ==============================================================================
# CELL 2: Configuration
# EDIT THIS CELL FOR YOUR VIDEOS
# ==============================================================================

import os
import torch
import imageio_ffmpeg

# ──────────────────────────────────────────────────────────────────────────────
# HUGGINGFACE TOKEN (REQUIRED for speaker diarization)
# Get from: https://huggingface.co/settings/tokens
# ──────────────────────────────────────────────────────────────────────────────
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))


# ──────────────────────────────────────────────────────────────────────────────
# VIDEO URLs AND LANGUAGES
# ──────────────────────────────────────────────────────────────────────────────
VIDEO_URLS = {https://www.youtube.com/watch?v=1N76UD-o95I
    "romania": "",
    "poland": "https://www.youtube.com/watch?v=WS8-dfk77yw"
}

VIDEO_LANGUAGES = {
    "romania": "ro",  # Romanian
    "poland": "pl"    # Polish
}

# ──────────────────────────────────────────────────────────────────────────────
# SPEAKER KEYWORDS (for identifying speakers after diarization)
# ──────────────────────────────────────────────────────────────────────────────
SPEAKER_KEYWORDS = {
    "romania": {
        "George Simion": ["george simion", "domnul simion", "candidatul simion", "sunt george", "domnule simion", "simion"],
        "Nicușor Dan": ["nicușor dan", "domnul dan", "candidatul dan", "sunt nicușor", "domnule dan", "nicușor", "dan"],
        "Moderator": ["andra miron", "monica mihai", "octav vasilescu", "sonia teodoriu", "jurnalist", "moderator", "doamna", "domnul moderator"]
    },
    "poland": {
        "Rafał Trzaskowski": ["trzaskowski", "rafal", "panie trzaskowski", "prezydent warszawy", "panie rafale", "rafale"],
        "Karol Nawrocki": ["nawrocki", "karol", "panie nawrocki", "szef ipn", "panie karolu"],
        "Moderator": ["jacek prusinowski", "panowie", "panie prezydencie", "moderator", "dziennikarz", "prowadzący"]
    }
}

# ──────────────────────────────────────────────────────────────────────────────
# OUTPUT FILE NAMING
# ──────────────────────────────────────────────────────────────────────────────
STEM_NAMES = {
    "romania": "RO_2025_Simion_Dan",
    "poland": "PL_2025_Nawrocki_Trzaskowski"
}

# ──────────────────────────────────────────────────────────────────────────────
# MODEL & DEVICE SETTINGS
# ──────────────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
BATCH_SIZE = 16 if DEVICE == "cuda" else 4
WHISPER_MODEL = "medium"  # Use "small" for 3-4x faster, slightly less accurate

# ──────────────────────────────────────────────────────────────────────────────
# FFMPEG PATH (Windows fix)
# ──────────────────────────────────────────────────────────────────────────────
FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir = os.path.dirname(FFMPEG_PATH)
if ffmpeg_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

# ──────────────────────────────────────────────────────────────────────────────
# PRINT CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────
print(f"🔧 Device: {DEVICE}")
print(f"🔧 Compute type: {COMPUTE_TYPE}")
print(f"🔧 Batch size: {BATCH_SIZE}")
print(f"🔧 Whisper model: {WHISPER_MODEL}")
print(f"🔧 FFmpeg: {FFMPEG_PATH}")
print(f"🔧 Videos to process: {list(VIDEO_URLS.keys())}")


🔧 Device: cpu
🔧 Compute type: int8
🔧 Batch size: 4
🔧 Whisper model: medium
🔧 FFmpeg: c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
🔧 Videos to process: ['romania', 'poland']


In [3]:
# ==============================================================================
# CELL 3: Checkpoint & Helper Functions
# NO NEED TO EDIT
# ==============================================================================

import json
import os
import torch
import torchaudio
import pandas as pd
from typing import Dict, List
from datetime import datetime

# ──────────────────────────────────────────────────────────────────────────────
# CHECKPOINT FUNCTIONS
# ──────────────────────────────────────────────────────────────────────────────
def save_checkpoint(data, filepath):
    """Save intermediate result to JSON file"""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    print(f"        💾 Checkpoint saved: {filepath}")

def load_checkpoint(filepath):
    """Load intermediate result from JSON file"""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def delete_checkpoint(filepath):
    """Delete checkpoint file after successful completion"""
    if os.path.exists(filepath):
        os.remove(filepath)
        print(f"        🗑️  Checkpoint deleted: {filepath}")

# ──────────────────────────────────────────────────────────────────────────────
# TIME FORMATTING
# ──────────────────────────────────────────────────────────────────────────────
def fmt_time(seconds: float) -> str:
    """Convert seconds to HH:MM:SS format"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# ──────────────────────────────────────────────────────────────────────────────
# SPEAKER IDENTIFICATION
# ──────────────────────────────────────────────────────────────────────────────
def identify_speaker(text: str, keywords: Dict[str, List[str]]) -> str:
    """Identify speaker based on keyword matching in text"""
    text_lower = text.lower()
    scores = {name: sum(1 for kw in kws if kw.lower() in text_lower) 
              for name, kws in keywords.items()}
    max_score = max(scores.values())
    if max_score == 0:
        return "Unknown"
    return max(scores, key=scores.get)

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO LOADING (avoids whisperx's broken subprocess call)
# ──────────────────────────────────────────────────────────────────────────────
def load_audio_file(audio_path: str) -> tuple:
    """
    Load audio file and return (audio_array, sample_rate).
    Uses torchaudio instead of whisperx's broken subprocess call.
    """
    waveform, sample_rate = torchaudio.load(audio_path)
    
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # Convert to numpy
    audio = waveform.squeeze().numpy()
    
    return audio, sample_rate

# ──────────────────────────────────────────────────────────────────────────────
# HF TOKEN VALIDATION
# ──────────────────────────────────────────────────────────────────────────────
def validate_hf_token(token: str) -> bool:
    """Validate HuggingFace token before processing"""
    try:
        import requests
        response = requests.get(
            "https://huggingface.co/api/whoami-v2",
            headers={"Authorization": f"Bearer {token}"}
        )
        if response.status_code == 200:
            print(f"✅ HF Token valid: {response.json()['name']}")
            return True
        else:
            print(f"❌ HF Token invalid! Status: {response.status_code}")
            print("   Get a new one at: https://huggingface.co/settings/tokens")
            return False
    except Exception as e:
        print(f"⚠️  Could not validate token: {e}")
        return True  # Proceed anyway

print("✅ Helper functions loaded")


✅ Helper functions loaded


In [4]:
# ==============================================================================
# CELL 4: Download Audio Function
# NO NEED TO EDIT
# ==============================================================================

import subprocess
from yt_dlp import YoutubeDL

def download_audio(url: str, stem: str) -> str:
    """Download audio from YouTube URL"""
    mp3_path = f"{stem}_audio.mp3"
    
    if os.path.exists(mp3_path):
        print(f"  ⏭️  Audio exists: {mp3_path}")
        return mp3_path
    
    print(f"  ⬇️  Downloading audio...")
    
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{stem}_audio.%(ext)s",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "ffmpeg_location": FFMPEG_PATH,
        "quiet": True,
        "noprogress": True,
        "no_warnings": True,
    }
    
    try:
        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        # Handle alternative extensions
        if not os.path.exists(mp3_path):
            for ext in ["webm", "m4a", "wav"]:
                alt_path = f"{stem}_audio.{ext}"
                if os.path.exists(alt_path):
                    subprocess.run([
                        FFMPEG_PATH, "-y", "-i", alt_path,
                        "-codec:a", "libmp3lame", "-q:a", "2", mp3_path
                    ], check=True, capture_output=True)
                    os.remove(alt_path)
                    break
        
        print(f"  ✅ Audio downloaded: {mp3_path}")
        return mp3_path
    except Exception as e:
        print(f"  ❌ Download failed: {e}")
        raise

print("✅ Download function loaded")


✅ Download function loaded


In [5]:
# ==============================================================================
# CELL 5: Main Processing Function (WITH CHECKPOINTS)
# NO NEED TO EDIT - This is the fixed version
# ==============================================================================

import gc
import whisperx

def process_video(debate_key: str, url: str, stem: str, keywords: Dict[str, List[str]]):
    """Process a single video: download, transcribe, diarize, and save"""
    
    print(f"\n{'='*70}")
    print(f"📹 PROCESSING: {debate_key.upper()}")
    print(f"{'='*70}")
    
    # ──────────────────────────────────────────────────────────────────────────
    # CHECKPOINT PATHS
    # ──────────────────────────────────────────────────────────────────────────
    ckpt_step2 = f"{stem}_step2_transcribe.json"
    ckpt_step3 = f"{stem}_step3_align.json"
    ckpt_step4 = f"{stem}_step4_diarize.json"
    
    try:
        # Download audio
        audio_path = download_audio(url, stem)
        
        # ──────────────────────────────────────────────────────────────────────
        # CHECKPOINT: Try to resume from latest checkpoint
        # ──────────────────────────────────────────────────────────────────────
        result = None
        start_from_step = 1
        
        if os.path.exists(ckpt_step4):
            print(f"\n  ⏭️  Found step 4 checkpoint, loading...")
            result = load_checkpoint(ckpt_step4)
            start_from_step = 5  # Skip to DataFrame building
        elif os.path.exists(ckpt_step3):
            print(f"\n  ⏭️  Found step 3 checkpoint, loading...")
            result = load_checkpoint(ckpt_step3)
            start_from_step = 4
        elif os.path.exists(ckpt_step2):
            print(f"\n  ⏭️  Found step 2 checkpoint, loading...")
            result = load_checkpoint(ckpt_step2)
            start_from_step = 3
        else:
            print(f"\n  🆕  No checkpoints found, starting fresh...")
        
        # ──────────────────────────────────────────────────────────────────────
        # STEP 1: Load Model (always required)
        # ──────────────────────────────────────────────────────────────────────
        if start_from_step <= 1:
            print(f"\n  [1/4] Loading WhisperX model...")
            model = whisperx.load_model(WHISPER_MODEL, DEVICE, compute_type=COMPUTE_TYPE, 
                                        language=VIDEO_LANGUAGES[debate_key])
        else:
            print(f"\n  ⏭️  Skipping step 1 (model already loaded)")
            model = None  # Will reload if needed
        
        # ──────────────────────────────────────────────────────────────────────
        # STEP 2: Transcribe (checkpoint after)
        # ──────────────────────────────────────────────────────────────────────
        if start_from_step <= 2:
            print(f"  [2/4] Loading and transcribing audio...")
            audio, sample_rate = load_audio_file(audio_path)
            duration = len(audio) / sample_rate
            print(f"        Duration: {duration/60:.1f} minutes")
            print(f"        Sample rate: {sample_rate}")
            print(f"        Language: {VIDEO_LANGUAGES[debate_key]} (specified)")
            
            result = model.transcribe(audio, batch_size=BATCH_SIZE, language=VIDEO_LANGUAGES[debate_key])
            save_checkpoint(result, ckpt_step2)
            
            # Cleanup model
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            del model
        else:
            print(f"  ⏭️  Skipping step 2 (using checkpoint)")
            # Need to reload audio for后续 steps
            audio, sample_rate = load_audio_file(audio_path)
        
        # ──────────────────────────────────────────────────────────────────────
        # STEP 3: Align timestamps (checkpoint after)
        # ──────────────────────────────────────────────────────────────────────
        if start_from_step <= 3:
            print(f"  [3/4] Aligning timestamps...")
            align_model, meta = whisperx.load_align_model(language_code=VIDEO_LANGUAGES[debate_key], device=DEVICE)
            result = whisperx.align(result["segments"], align_model, meta, audio, DEVICE)
            save_checkpoint(result, ckpt_step3)
            
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            del align_model
        else:
            print(f"  ⏭️  Skipping step 3 (using checkpoint)")
        
        # ──────────────────────────────────────────────────────────────────────
        # STEP 4: Diarization (checkpoint after)
        # ──────────────────────────────────────────────────────────────────────
        if start_from_step <= 4:
            print(f"  [4/4] Running speaker diarization...")
            diarize_model = whisperx.DiarizationPipeline(token=HF_TOKEN, device=DEVICE)
            diarize_segments = diarize_model(audio, min_speakers=2, max_speakers=4)
            result = whisperx.assign_word_speakers(diarize_segments, result)
            save_checkpoint(result, ckpt_step4)
            
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            del diarize_model
        else:
            print(f"  ⏭️  Skipping step 4 (using checkpoint)")
        
        # ──────────────────────────────────────────────────────────────────────
        # STEP 5: Build DataFrame and Save
        # ──────────────────────────────────────────────────────────────────────
        print(f"\n  📊 Building transcript DataFrame...")
        rows = []
        for seg in result.get("segments", []):
            text = seg.get("text", "").strip()
            if text:
                rows.append({
                    "start": seg.get("start", 0),
                    "end": seg.get("end", 0),
                    "start_formatted": fmt_time(seg.get("start", 0)),
                    "end_formatted": fmt_time(seg.get("end", 0)),
                    "speaker": seg.get("speaker", "SPEAKER_0"),
                    "text": text
                })
        
        df = pd.DataFrame(rows)
        
        if df.empty:
            print("  ❌ No segments found!")
            return None
        
        print(f"  🎯 Identifying speakers...")
        speaker_texts = df.groupby("speaker")["text"].apply(lambda x: " ".join(x)).to_dict()
        
        speaker_mapping = {}
        for spk_id, combined_text in speaker_texts.items():
            speaker_mapping[spk_id] = identify_speaker(combined_text, keywords)
            print(f"        {spk_id} → {speaker_mapping[spk_id]}")
        
        df["speaker_name"] = df["speaker"].map(speaker_mapping)
        
        print(f"  🔗 Merging consecutive turns...")
        merged = []
        prev = None
        
        for _, row in df.iterrows():
            if prev and row["speaker_name"] == prev["speaker_name"]:
                prev["end"] = row["end"]
                prev["end_formatted"] = row["end_formatted"]
                prev["text"] += " " + row["text"]
            else:
                if prev:
                    merged.append(prev)
                prev = row.to_dict()
        
        if prev:
            merged.append(prev)
        
        df_final = pd.DataFrame(merged)
        
        print(f"\n  💾 Saving files...")
        csv_path = f"{stem}_transcript.csv"
        df_final.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print(f"        ✅ {csv_path}")
        
        txt_path = f"{stem}_transcript.txt"
        with open(txt_path, "w", encoding="utf-8") as f:
            for _, row in df_final.iterrows():
                f.write(f"[{row['start_formatted']} → {row['end_formatted']}] {row['speaker_name']}: {row['text']}\n")
        print(f"        ✅ {txt_path}")
        
        # Cleanup checkpoints after successful completion
        print(f"\n  🧹 Cleaning up checkpoints...")
        delete_checkpoint(ckpt_step2)
        delete_checkpoint(ckpt_step3)
        delete_checkpoint(ckpt_step4)
        
        print(f"\n{'='*70}")
        print(f"✅ COMPLETE: {debate_key.upper()}")
        print(f"  Total segments: {len(df_final)}")
        print(f"  Unique speakers: {df_final['speaker_name'].nunique()}")
        for name in df_final['speaker_name'].unique():
            count = len(df_final[df_final['speaker_name']==name])
            print(f"    - {name}: {count} segments")
        print(f"{'='*70}\n")
        
        return df_final
        
    except Exception as e:
        print(f"\n  ❌ ERROR processing {debate_key}: {e}")
        import traceback
        traceback.print_exc()
        print(f"\n  ⚠️  Checkpoints saved! Re-run this cell to resume.")
        return None

print("✅ Process function loaded (WITH CHECKPOINTS)")


✅ Process function loaded (WITH CHECKPOINTS)


In [6]:
# ==============================================================================
# CELL 6: Run All Videos
# RUN THIS TO START PROCESSING
# ==============================================================================

from datetime import datetime

print("\n" + "█"*70)
print("🚀 STARTING YOUTUBE TRANSCRIPT EXTRACTION")
print(f"   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("█"*70)

# Validate HF token first
print("\n🔐 Validating HuggingFace token...")
validate_hf_token(HF_TOKEN)

results = {}

for debate_key, url in VIDEO_URLS.items():
    try:
        results[debate_key] = process_video(
            debate_key=debate_key,
            url=url,
            stem=STEM_NAMES[debate_key],
            keywords=SPEAKER_KEYWORDS[debate_key]
        )
    except KeyboardInterrupt:
        print(f"\n⚠️  Interrupted by user. Checkpoints saved for {debate_key}.")
        print("   Re-run Cell 6 to resume.")
        break
    except Exception as e:
        print(f"\n❌ FAILED {debate_key}: {e}")
        results[debate_key] = None

print("\n" + "█"*70)
print("📋 FINAL SUMMARY")
print("█"*70)

for debate_key, df in results.items():
    if df is not None:
        print(f"\n{debate_key.upper()}:")
        print(df[["start_formatted", "end_formatted", "speaker_name", "text"]].head(10).to_string())
        print(f"\n  Output files:")
        print(f"    - {STEM_NAMES[debate_key]}_transcript.csv")
        print(f"    - {STEM_NAMES[debate_key]}_transcript.txt")
    else:
        print(f"\n{debate_key.upper()}: ❌ FAILED (checkpoints may exist)")

print("\n" + "█"*70)
print("🎉 ALL DONE!")
print(f"   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("█"*70 + "\n")


██████████████████████████████████████████████████████████████████████
🚀 STARTING YOUTUBE TRANSCRIPT EXTRACTION
   2026-05-22 02:05:18
██████████████████████████████████████████████████████████████████████

🔐 Validating HuggingFace token...
✅ HF Token valid: Rzwnn

📹 PROCESSING: ROMANIA
  ⏭️  Audio exists: RO_2025_Simion_Dan_audio.mp3

  🆕  No checkpoints found, starting fresh...

  [1/4] Loading WhisperX model...


c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.8.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github

2026-05-22 02:06:13 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisperx\assets\pytorch_model.bin`


  [2/4] Loading and transcribing audio...


...

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisperx\assets\pytorch_model.bin`
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


  [2/4] Loading and transcribing audio...
        Duration: 114.0 minutes
        Sample rate: 48000
        Language: pl (specified)
        💾 Checkpoint saved: PL_2025_Nawrocki_Trzaskowski_step2_transcribe.json
  [3/4] Aligning timestamps...


c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\.cache\huggingface\hub\models--jonatasgrosman--wav2vec2-large-xlsr-53-polish. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


        💾 Checkpoint saved: PL_2025_Nawrocki_Trzaskowski_step3_align.json
  [4/4] Running speaker diarization...

  ❌ ERROR processing poland: module 'whisperx' has no attribute 'DiarizationPipeline'

  ⚠️  Checkpoints saved! Re-run this cell to resume.

██████████████████████████████████████████████████████████████████████
📋 FINAL SUMMARY
██████████████████████████████████████████████████████████████████████

ROMANIA: ❌ FAILED (checkpoints may exist)

POLAND: ❌ FAILED (checkpoints may exist)

██████████████████████████████████████████████████████████████████████
🎉 ALL DONE!
   2026-05-23 00:17:13
██████████████████████████████████████████████████████████████████████



Traceback (most recent call last):
  File "C:\Users\andre\AppData\Local\Temp\ipykernel_13164\2609434227.py", line 104, in process_video
    diarize_model = whisperx.DiarizationPipeline(token=HF_TOKEN, device=DEVICE)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: module 'whisperx' has no attribute 'DiarizationPipeline'
